# Korea Pine-Wilt Fisher-KPP Simulation

This notebook loads the compact Korea Forest Service pine-wilt observation data committed in this repository, builds yearly density grids, and runs a 2D Fisher-KPP RK4 forward simulation from the 2016 observed density. The raw annual CSV files are tracked by checksum in the manifest because several source files exceed normal GitHub blob limits.

## 1. Setup

In Colab, this cell clones or refreshes the GitHub repository under `/content/fisher-pinn`. Locally, run the notebook from the repository root or any child directory.

In [ ]:
%matplotlib inline

from __future__ import annotations

import csv
import json
import subprocess
import sys
from argparse import Namespace
from pathlib import Path

from IPython.display import Image, Markdown, display

REPO_URL = "https://github.com/rladbsco24/fisher-pinn.git"
REPO_BRANCH = "main"


def _has_project(root: Path) -> bool:
    return (
        (root / "fisher_origin_lab").exists()
        and (root / "data" / "korea_pine_wilt" / "processed" / "manifest.json").exists()
    )


def _run_git(args: list[str], cwd: Path | None = None) -> None:
    subprocess.run(["git", *args], cwd=str(cwd) if cwd else None, check=True)


def _prepare_colab_repo(repo_dir: Path) -> Path:
    if not repo_dir.exists():
        _run_git(["clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
    elif (repo_dir / ".git").exists():
        _run_git(["fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo_dir)
        _run_git(["checkout", "--force", "FETCH_HEAD"], cwd=repo_dir)
    return repo_dir.resolve()


def _ensure_python_deps(root: Path) -> None:
    missing = []
    for module in ["pyproj", "PIL"]:
        try:
            __import__(module)
        except ImportError:
            missing.append(module)
    if missing:
        print(f"installing missing notebook dependencies: {missing}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")], check=True)


PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if _has_project(candidate):
        PROJECT_ROOT = candidate
        break

if not _has_project(PROJECT_ROOT) and Path("/content").exists():
    PROJECT_ROOT = _prepare_colab_repo(Path("/content/fisher-pinn"))

if not _has_project(PROJECT_ROOT):
    raise RuntimeError(
        "Project files were not found. Run this notebook from the fisher-pinn repository root "
        "or use Colab with network access so the repository can be cloned."
    )

_ensure_python_deps(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"project root: {PROJECT_ROOT}")


## 2. Configuration

The defaults are intended to run in a few minutes on a typical Colab or local CPU session. Increase `GRID_SIZE` and `STEPS_PER_YEAR` for a finer RK4 baseline.

In [ ]:
from fisher_origin_lab.korea_data import load_manifest, load_korea_pine_wilt_points, korea_physics_prior_from_physical
from scripts.run_korea_pine_wilt_simulation import run as run_korea_pine_wilt_simulation

OUT_DIR = PROJECT_ROOT / "runs" / "korea_pine_wilt_notebook"
GRID_SIZE = 96
PARAMETERIZATION = "physical"
PHYSICS_LENGTH_SCALE = "max_extent"
DIFFUSION_KM2_PER_YEAR = 15.5
REACTION_PER_YEAR = 0.70
# These normalized values are used only if PARAMETERIZATION = "normalized".
DIFFUSION = 0.0015
REACTION = 0.70
STEPS_PER_YEAR = 80
END_YEAR = 2030
MAP_GIF_FPS = 1.2
MAP_GIF_MAX_FRAMES = 15

# The PINN baseline is intentionally modest so the notebook remains Colab-friendly.
# Increase this for a stronger fit, but interpret it as a diagnostic baseline, not a calibrated forecast.
PINN_EPOCHS = 120
PINN_COLLOCATION_POINTS = 768
PINN_BOUNDARY_POINTS = 128
PINN_BATCH_SIZE = 4096
PINN_INITIAL_CONDITION_WEIGHT = 16.0
PINN_INITIAL_CONDITION_POINTS = 2048
PINN_SEA_WEIGHT = 2.0
PINN_INITIAL_REACTION = None
PINN_PHYSICS_ANCHOR_WEIGHT = 0.08
PINN_COEFFICIENT_FIELD_WEIGHT = 0.02

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"output dir: {OUT_DIR}")


## 3. Dataset Manifest And Integrity

The compact files contain infected-tree coordinates and observation year only. The manifest records raw CSV sizes and sha256 checksums so the source bundle can be audited separately.

In [ ]:
manifest = load_manifest()
compact = manifest["compact_files"]
raw_by_year = {entry["year"]: entry for entry in manifest["raw_files"]}

data_root = PROJECT_ROOT / "data" / "korea_pine_wilt"
csv_gz_path = data_root / compact["csv_gzip"]
npz_path = data_root / compact["npz"]
for data_path in [csv_gz_path, npz_path]:
    if not data_path.exists():
        raise FileNotFoundError(f"Required compact data file is missing: {data_path}")

expected_records = sum(int(value) for value in compact["year_counts"].values())
if int(compact["records"]) != expected_records:
    raise ValueError("Manifest record count does not match year_counts sum.")

rows = ["| year | compact records | raw CSV MB | raw sha256 prefix |", "|---:|---:|---:|---|"]
for year, count in compact["year_counts"].items():
    raw = raw_by_year[int(year)]
    rows.append(f"| {year} | {int(count):,} | {raw['bytes'] / 1_000_000:.1f} | `{raw['sha256'][:12]}` |")

display(Markdown("\n".join(rows)))
print(json.dumps(manifest["source"], indent=2, ensure_ascii=False))
print(f"compact csv.gz: {csv_gz_path} ({csv_gz_path.stat().st_size / 1_000_000:.1f} MB)")
print(f"compact npz:    {npz_path} ({npz_path.stat().st_size / 1_000_000:.1f} MB)")


## 4. Load Compact Observation Points

The coordinate reference system is `EPSG:5179`. This step loads 3.18 million compact observation points into memory from the committed NPZ file.

In [ ]:
points = load_korea_pine_wilt_points()
print(f"records: {len(points.year):,}")
print(f"years: {int(points.year.min())}..{int(points.year.max())}")
print(f"x range: {points.x.min():.1f}..{points.x.max():.1f}")
print(f"y range: {points.y.min():.1f}..{points.y.max():.1f}")
print(f"crs: {points.crs}")

## 5. Run RK4 And PINN Baselines

The 2016 observed density grid is used as the RK4 initial condition and as a separate soft initial-condition anchor for the PINN at `t=0`. The physical scalar diffusion is converted to anisotropic normalized `D_x` and `D_y` because Korea's projected x/y extents are different. Both RK4 and PINN use the same land mask, sea-exclusion constraint, observed-year metrics, physical D/r prior, and a smooth bounded spatial coefficient-field option for the PINN. The PINN baseline is a diagnostic comparison, not a calibrated operational disease forecast.


In [ ]:
summary = run_korea_pine_wilt_simulation(
    Namespace(
        grid_size=GRID_SIZE,
        pad_m=15_000.0,
        capacity_percentile=99.0,
        smooth_passes=1,
        parameterization=PARAMETERIZATION,
        physics_length_scale=PHYSICS_LENGTH_SCALE,
        diffusion_km2_per_year=DIFFUSION_KM2_PER_YEAR,
        reaction_per_year=REACTION_PER_YEAR,
        diffusion=DIFFUSION,
        reaction=REACTION,
        steps_per_year=STEPS_PER_YEAR,
        end_year=END_YEAR,
        output_dir=OUT_DIR,
        skip_pinn=False,
        pinn_epochs=PINN_EPOCHS,
        pinn_batch_size=PINN_BATCH_SIZE,
        pinn_collocation_points=PINN_COLLOCATION_POINTS,
        pinn_boundary_points=PINN_BOUNDARY_POINTS,
        pinn_lr=2.0e-3,
        pinn_data_weight=8.0,
        pinn_pde_weight=0.05,
        pinn_boundary_weight=0.01,
        pinn_initial_condition_weight=PINN_INITIAL_CONDITION_WEIGHT,
        pinn_initial_condition_points=PINN_INITIAL_CONDITION_POINTS,
        pinn_sea_weight=PINN_SEA_WEIGHT,
        pinn_initial_reaction=PINN_INITIAL_REACTION,
        pinn_physics_anchor_weight=PINN_PHYSICS_ANCHOR_WEIGHT,
        pinn_coefficient_field_weight=PINN_COEFFICIENT_FIELD_WEIGHT,
        map_gif_fps=MAP_GIF_FPS,
        map_gif_max_frames=MAP_GIF_MAX_FRAMES,
        seed=7,
    )
)
print(json.dumps(summary, indent=2, ensure_ascii=False))


## 6. Observed-Year Baseline Metrics

The table compares RK4 and the repository PINN baseline against observed density grids for 2016-2023. In addition to relative L2 and correlation, it reports mass error and threshold-support errors so near-zero collapse or haze-like false positives are visible.


In [ ]:
metrics_path = OUT_DIR / "korea_pine_wilt_baseline_metrics.csv"
with metrics_path.open("r", encoding="utf-8", newline="") as f:
    metric_rows = list(csv.DictReader(f))


def _float_cell(row, key, default="nan"):
    value = row.get(key, default)
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


md_rows = [
    "| method | year | relative L2 | correlation | mass abs. err | FN@0.05 | Dice@0.05 | observed mean | simulated mean |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for row in metric_rows:
    md_rows.append(
        "| {method} | {year} | {l2:.4f} | {corr:.4f} | {mass:.4f} | {fnr:.4f} | {dice:.4f} | {obs:.4f} | {sim:.4f} |".format(
            method=row["method"],
            year=row["year"],
            l2=_float_cell(row, "relative_l2"),
            corr=_float_cell(row, "correlation"),
            mass=_float_cell(row, "mass_absolute_error"),
            fnr=_float_cell(row, "support_fnr_005"),
            dice=_float_cell(row, "support_dice_005"),
            obs=_float_cell(row, "observed_mean"),
            sim=_float_cell(row, "simulated_mean"),
        )
    )

display(Markdown("\n".join(md_rows)))

baseline_summary_rows = [
    "| method | mean relative L2 | mean correlation | mean mass abs. err | mean FN@0.05 | mean Dice@0.05 | status |",
    "|---|---:|---:|---:|---:|---:|---|",
]
for method, info in summary["baselines"].items():
    baseline_summary_rows.append(
        "| {method} | {l2:.4f} | {corr:.4f} | {mass:.4f} | {fnr:.4f} | {dice:.4f} | {status} |".format(
            method=method,
            l2=float(info.get("mean_relative_l2_observed_years") or float("nan")),
            corr=float(info.get("mean_correlation_observed_years") or float("nan")),
            mass=float(info.get("mean_mass_absolute_error") or float("nan")),
            fnr=float(info.get("mean_support_fnr_005") or float("nan")),
            dice=float(info.get("mean_support_dice_005") or float("nan")),
            status=info.get("status", "fixed_solver"),
        )
    )
display(Markdown("\n".join(baseline_summary_rows)))

display(Markdown("### Baseline protocol"))
display(Markdown("```json\n" + json.dumps(summary["baseline_protocol"], indent=2, ensure_ascii=False) + "\n```"))


## 7. Visual Outputs

The script writes the PNG files displayed below into `runs/korea_pine_wilt_notebook`. They include gridded observations, RK4 forecast, PINN observed-year reconstruction, and RK4-vs-PINN metric comparison.

In [ ]:
for image_name in [
    "observed_density_by_year.png",
    "rk4_forecast_timeline.png",
    "pinn_baseline_observed_years.png",
    "baseline_metric_comparison.png",
    "observed_vs_simulated_metrics.png",
    "korea_map_baselines_preview.png",
    "korea_map_baselines.gif",
]:
    image_path = OUT_DIR / image_name
    display(Markdown(f"### `{image_name}`"))
    display(Image(filename=str(image_path)))


## 8. Notes

This notebook is a reproducible bridge from the Korea Forest Service observation coordinates to a Fisher-KPP forward baseline. The default Korea diffusion is specified in physical units (km^2/year), converted to normalized anisotropic `D_x`/`D_y`, and used consistently by RK4 and the PINN PDE residual. The PINN can additionally learn a smooth bounded correction field for `D(x,y)` and `r(x,y)`, regularized toward the constant-coefficient prior. A paper-grade real-data model would still need reporting intensity, control actions, detection bias, forest or terrain masks, yearly policy changes, and spatially varying diffusion/reaction terms.
